# IRWA Final Project — Part 1: Text Processing and Exploratory Data Analysis

In [36]:
import sys
import collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from wordcloud import WordCloud

# project root = first parent folder containing the `myapp` package
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "myapp").is_dir())
sys.path.insert(0, str(ROOT))

from myapp.search.load_corpus import load_corpus
from myapp.search.preprocessing import (build_terms, normalize_text, tokenize, details_to_text, preprocess_corpus,
                                        STOP_WORDS, STEMMER, DETAIL_KEYS)

DATA_PATH = ROOT / "data" / "fashion_products_dataset.json"
LABELS_PATH = ROOT / "data" / "validation_labels.csv"
PROCESSED_PATH = ROOT / "data" / "processed_corpus.pkl"
FIG_DIR = ROOT / "project_progress" / "part_1" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

---
# PART 1: Data preparation

## 1.1 Load the corpus

In [37]:
# raw: original JSON (strings); corpus: pid -> Document with parsed prices/discount/rating
raw = pd.read_json(DATA_PATH)
corpus = load_corpus(str(DATA_PATH))
print("documents:", len(raw), "| unique pid:", raw.pid.nunique())
raw.head(3)

documents: 28080 | unique pid: 28080


,_id,actual_price,average_rating,brand,category,crawled_at,description,discount,images,out_of_stock,pid,product_details,seller,selling_price,sub_category,title,url
0,fa8e22d6-c0b6-5229-bb9e-ad52eda39a0a,"2,999",3.9,York,Clothing and Accessories,2021-02-10 20:11:51,Yorker trackpants made from 100% rich combed cotton giving it a rich look.De...,69% off,[https://rukminim1.flixcart.com/image/128/128/jr3t5e80/track-pant/z/y/n/m-10...,False,TKPFCZ9EA7H5FYZH,"[{'Style Code': '1005COMBO2'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Poc...",Shyam Enterprises,921,Bottomwear,Solid Women Multicolor Track Pants,https://www.flipkart.com/yorker-solid-men-multicolor-track-pants/p/itmd2c76a...
1,893e6980-f2a0-531f-b056-34dd63fe912c,"1,499",3.9,York,Clothing and Accessories,2021-02-10 20:11:52,Yorker trackpants made from 100% rich combed cotton giving it a rich look.De...,66% off,[https://rukminim1.flixcart.com/image/128/128/kfyasnk0/track-pant/g/5/y/s-19...,False,TKPFCZ9EJZV2UVRZ,"[{'Style Code': '1005BLUE'}, {'Closure': 'Drawstring, Elastic'}, {'Pockets':...",Shyam Enterprises,499,Bottomwear,Solid Men Blue Track Pants,https://www.flipkart.com/yorker-solid-men-blue-track-pants/p/itmfczez7v6rzwe...
2,eb4c8eab-8206-59d0-bcd1-a724d96bf74f,"2,999",3.9,York,Clothing and Accessories,2021-02-10 20:11:52,Yorker trackpants made from 100% rich combed cotton giving it a rich look.De...,68% off,[https://rukminim1.flixcart.com/image/128/128/jr3t5e80/track-pant/e/c/s/xl-1...,False,TKPFCZ9EHFCY5Z4Y,"[{'Style Code': '1005COMBO4'}, {'Closure': 'Elastic'}, {'Pockets': 'Side Poc...",Shyam Enterprises,931,Bottomwear,Solid Men Multicolor Track Pants,https://www.flipkart.com/yorker-solid-men-multicolor-track-pants/p/itm1eee3c...


## 1.2 Text pre-processing pipeline

In [38]:
# step-by-step pre-processing of one product (title + description)
example = corpus["TKPFCZ9EA7H5FYZH"]
text = example.title + " " + example.description
normalized = normalize_text(text)
tokens = tokenize(normalized)
no_stop = [t for t in tokens if t not in STOP_WORDS]
stemmed = [STEMMER.stem(t) for t in no_stop]

print("RAW:        ", text, "\n")
print("NORMALIZED: ", normalized, "\n")
print("TOKENS:     ", tokens, "\n")
print("NO STOP:    ", no_stop, "\n")
print("STEMMED:    ", stemmed)

RAW:         Solid Women Multicolor Track Pants Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India 

NORMALIZED:  solid women multicolor track pants yorker trackpants made from 100  rich combed cotton giving it a rich look designed for comfort skin friendly fabric itch free waistband   great for all year round use proudly made in india 

TOKENS:      ['solid', 'women', 'multicolor', 'track', 'pants', 'yorker', 'trackpants', 'made', 'from', 'rich', 'combed', 'cotton', 'giving', 'it', 'rich', 'look', 'designed', 'for', 'comfort', 'skin', 'friendly', 'fabric', 'itch', 'free', 'waistband', 'great', 'for', 'all', 'year', 'round', 'use', 'proudly', 'made', 'in', 'india'] 

NO STOP:     ['solid', 'women', 'multicolor', 'track', 'pants', 'yorker', 'trackpants', 'made', 'rich', 'combed', 'cotton', 'giving', 'rich', 'look', 'designed', 'comfort', 'skin', 'frie

In [39]:
# the same function is applied to queries
for q in ["women full sleeve sweatshirt cotton", "men slim jeans blue", "Men's Slim-Fit T-Shirts", "ladies polycotton tees"]:
    print(f"{q!r:40} -> {build_terms(q)}")

'women full sleeve sweatshirt cotton'    -> ['women', 'full', 'sleev', 'sweatshirt', 'cotton']
'men slim jeans blue'                    -> ['men', 'slim', 'jean', 'blue']
"Men's Slim-Fit T-Shirts"                -> ['men', 'slim', 'fit', 'tshirt']
'ladies polycotton tees'                 -> ['women', 'poli', 'cotton', 'tee']


### Effect of each step on the vocabulary

In [40]:
# vocabulary size and number of tokens after each step, over title + description
texts = (raw.title + " " + raw.description.fillna("")).tolist()

steps = {
    "raw (whitespace split)": lambda t: t.split(),
    "+ lowercase, normalization, punctuation, tokenization": lambda t: tokenize(normalize_text(t)),
    "+ stop-word removal": lambda t: build_terms(t, stemming=False),
    "+ stemming": lambda t: build_terms(t),
}
rows = []
for name, fn in steps.items():
    toks = [tok for t in texts for tok in fn(t)]
    rows.append({"step": name, "tokens": len(toks), "vocabulary": len(set(toks))})
vocab_steps = pd.DataFrame(rows)
vocab_steps

,step,tokens,vocabulary
0,raw (whitespace split),1017425,13521
1,"+ lowercase, normalization, punctuation, tokenization",986146,6325
2,+ stop-word removal,687984,6205
3,+ stemming,687984,4741


## 1.3 Pre-process the whole corpus

In [ ]:
# one row per document, saved to be loaded by the indexing in Part 2
processed = preprocess_corpus(corpus)
processed.to_pickle(PROCESSED_PATH)
print(processed.shape, "->", PROCESSED_PATH)
processed[["pid", "title", "title_terms", "details_terms", "brand_terms", "sub_category_terms"]].head()

(28080, 21) -> /Users/bielmongeortiz/Desktop/UNI/4t Curs/1r Trimestre/Recuperació de la Informació i Anàlisi de la Web/Practices/irwa-final-project/data/processed_corpus.pkl


,pid,title,title_terms,details_terms,brand_terms,sub_category_terms
TKPFCZ9EA7H5FYZH,TKPFCZ9EA7H5FYZH,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant]","[cotton, blend, solid, multicolor, elast]",[york],[bottomwear]
TKPFCZ9EJZV2UVRZ,TKPFCZ9EJZV2UVRZ,Solid Men Blue Track Pants,"[solid, men, blue, track, pant]","[cotton, blend, solid, blue, drawstr, elast]",[york],[bottomwear]
TKPFCZ9EHFCY5Z4Y,TKPFCZ9EHFCY5Z4Y,Solid Men Multicolor Track Pants,"[solid, men, multicolor, track, pant]","[cotton, blend, solid, multicolor, elast]",[york],[bottomwear]
TKPFCZ9ESZZ7YWEF,TKPFCZ9ESZZ7YWEF,Solid Women Multicolor Track Pants,"[solid, women, multicolor, track, pant]","[cotton, blend, solid, multicolor, elast]",[york],[bottomwear]
TKPFCZ9EVXKBSUD7,TKPFCZ9EVXKBSUD7,"Solid Women Brown, Grey Track Pants","[solid, women, brown, grey, track, pant]","[cotton, blend, solid, brown, grey, drawstr, elast]",[york],[bottomwear]


## 1.4 Handling `category`, `sub_category`, `brand`, `product_details` and `seller`

In [ ]:
# how distinctive is each field, if the most frequent value covers most documents, its IDF is ~0
def field_profile(series):
    s = series.dropna().astype(str)
    s = s[s.str.strip() != ""]
    return {"non-empty %": round(100 * len(s) / len(series), 1), "distinct values": s.nunique(),
            "most frequent": s.value_counts().index[0], "most frequent %": round(100 * s.value_counts().iloc[0] / len(s), 1)}

details_text = processed.product_details.apply(details_to_text)
profile = pd.DataFrame({
    "category": field_profile(raw.category),
    "sub_category": field_profile(raw.sub_category),
    "brand": field_profile(raw.brand),
    "seller": field_profile(processed.seller),
    "product_details (selected values)": field_profile(details_text),
}).T
profile

,non-empty %,distinct values,most frequent,most frequent %
category,100.0,4,Clothing and Accessories,96.6
sub_category,100.0,24,Topwear,54.5
brand,92.8,324,ECKO Unl,3.6
seller,94.1,498,RetailNet,5.4
product_details (selected values),98.1,10359,Cotton Blend Solid Half Sleeve Polo Neck Men Regular T Shirts,0.9


In [43]:
# % of labelled documents (validation_labels.csv) with each query term in each field
labels = pd.read_csv(LABELS_PATH)
QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
FIELDS = ["title_terms", "description_terms", "details_terms", "sub_category_terms", "brand_terms"]
lab = labels.merge(processed.drop(columns="title"), on="pid")

rows = []
for qid, query in QUERIES.items():
    sub = lab[lab.query_id == qid]
    for term in build_terms(query):
        row = {"query": qid, "term": term}
        for f in FIELDS:
            row[f.replace("_terms", "")] = f"{100 * sub[f].apply(lambda ts: term in ts).mean():.0f}%"
        rows.append(row)
print("% of labelled documents containing the query term in each field")
pd.DataFrame(rows)

% of labelled documents containing the query term in each field


,query,term,title,description,details,sub_category,brand
0,1,women,100%,40%,20%,0%,0%
1,1,full,80%,50%,80%,0%,0%
2,1,sleev,80%,50%,100%,0%,0%
3,1,sweatshirt,80%,65%,20%,0%,0%
4,1,cotton,0%,100%,90%,0%,0%
5,2,men,45%,0%,45%,0%,0%
6,2,slim,60%,45%,10%,0%,0%
7,2,jean,100%,65%,70%,0%,0%
8,2,blue,100%,40%,100%,0%,0%


In [ ]:
# AND relevant / non-relevant labelled documents retrieved per set of indexed fields
def and_match(doc_terms, query_terms):
    return set(query_terms) <= set(doc_terms)

field_sets = {
    "title only": ["title_terms"],
    "title + description": ["title_terms", "description_terms"],
    "title + description + details": ["title_terms", "description_terms", "details_terms"],
    "all fields (terms)": ["terms"],
}
rows = []
for qid, query in QUERIES.items():
    q = build_terms(query)
    sub = lab[lab.query_id == qid]
    for name, fields in field_sets.items():
        matched = sub.apply(lambda r: and_match(sum((r[f] for f in fields), []), q), axis=1)
        rows.append({"query": qid, "indexed fields": name,
                     "relevant docs matched": f"{(matched & (sub.labels == 1)).sum()}/{(sub.labels == 1).sum()}",
                     "non-relevant docs matched": f"{(matched & (sub.labels == 0)).sum()}/{(sub.labels == 0).sum()}"})
pd.DataFrame(rows)

,query,indexed fields,relevant docs matched,non-relevant docs matched
0,1,title only,0/13,0/7
1,1,title + description,13/13,7/7
2,1,title + description + details,13/13,7/7
3,1,all fields (terms),13/13,7/7
4,2,title only,4/10,1/10
5,2,title + description,6/10,2/10
6,2,title + description + details,6/10,2/10
7,2,all fields (terms),6/10,2/10


In [45]:
# documents returned by each AND query over the whole corpus, per set of indexed fields
rows = []
for qid, query in QUERIES.items():
    q = build_terms(query)
    row = {"query": query}
    for name, fields in field_sets.items():
        row[name] = processed.apply(lambda r: and_match(sum((r[f] for f in fields), []), q), axis=1).sum()
    rows.append(row)
pd.DataFrame(rows).set_index("query")

,title only,title + description,title + description + details,all fields (terms)
query,,,,
women full sleeve sweatshirt cotton,0,241,549,549
men slim jeans blue,116,176,217,217


## 1.5 Handling `out_of_stock`, `selling_price`, `discount`, `actual_price`, `average_rating`

In [46]:
# missing values after parsing, and reported discount vs discount derived from the prices
NUM = ["selling_price", "actual_price", "discount", "average_rating"]
print("missing values after parsing:")
print(processed[NUM].isna().sum().to_string(), "\n")

# consistency check: reported discount vs discount derived from prices
derived = 100 * (1 - processed.selling_price / processed.actual_price)
reported = pd.to_numeric(raw.set_index("pid").discount.str.extract(r"(\d+)")[0], errors="coerce").reindex(processed.index)
diff = (derived - reported).abs()
print(f"reported vs derived discount, |diff| <= 1 point: {(diff <= 1).mean():.1%} of documents with both values")
print(f"discounts filled from prices: {(reported.isna() & processed.discount.notna()).sum()}")
processed[NUM].describe().round(2)

missing values after parsing:
selling_price        2
actual_price       777
discount           777
average_rating    2261 

reported vs derived discount, |diff| <= 1 point: 96.9% of documents with both values
discounts filled from prices: 78


,selling_price,actual_price,discount,average_rating
count,28078.00,27303.00,27303.00,25819.00
mean,705.64,1455.53,50.11,3.63
std,549.68,939.98,17.07,0.66
min,99.00,150.00,0.00,1.00
25%,390.00,849.00,40.00,3.20
50%,545.00,1199.00,52.00,3.80
75%,820.00,1799.00,63.00,4.10
max,7999.00,12999.00,87.00,5.00
